# CLV Thesis — Kaggle GPU Runner

**Purpose:** Run the deep-learning training sweep (LSTM + Transformer models) on Kaggle's free T4 GPU without the 8–14 h local runtime.

**Kaggle basics you need to know before running this notebook:**

| Concept | What it means |
|---------|---------------|
| `/kaggle/input/<slug>/` | Where your uploaded datasets are mounted — **read-only**. You cannot write here. |
| `/kaggle/working/` | The writable scratch space for this session. Outputs saved here survive when you download them. |
| Session limit | T4 GPU notebooks get **12 hours** of compute. After that the kernel dies. |
| Persistence | Code edits in the notebook **are saved** to your Kaggle account. Output files in `/kaggle/working/` must be explicitly downloaded or they disappear at session end. |
| Internet access | Off by default — turn it on in **Notebook settings → Internet** if you need `pip install` to reach PyPI. |

---

## Before you start: one-time Kaggle setup

### 1 · Upload your raw datasets

Each dataset must be uploaded to Kaggle as a **Dataset** so it is mounted at a stable path.  
Go to [kaggle.com/datasets/new](https://www.kaggle.com/datasets/new) for each one.

**Important:** Kaggle requires dataset titles to be at least 7 characters and derives the URL slug directly from the title. Type the slug value exactly as the title — all lowercase, no spaces.

| Files to upload | **Title to use (becomes the slug)** | Mounted at |
|----------------|--------------------------------------|------------|
| `CDNOW_sample.txt` required; `CDNOW_master.txt` optional for Cell 2.5 | `cdnow-dataset` | `/kaggle/input/cdnow-dataset/` |
| `online_retail_II.xlsx` | `uci-retail` | `/kaggle/input/uci-retail/` |
| `ta_feng_all_months_merged.csv` | `tafeng-dataset` | `/kaggle/input/tafeng-dataset/` |
| `transactions.csv`, `hh_demographic.csv`, `campaign_table.csv`, `coupon_redempt.csv` | `dunnhumby` | `/kaggle/input/dunnhumby/` |

> The code in `src/utils/config.py` automatically maps these slugs back to the short  
> internal names (`cdnow`, `uci`, `tafeng`, `dunnhumby`) that your YAML configs use.  
> You do not need to rename anything in the repo.

### 2 · Add your datasets to this notebook

Open this notebook on Kaggle → right-hand panel → **"+ Add Data"** → search for each dataset you just uploaded → click **Add**.  
After adding, each dataset appears at its `/kaggle/input/<slug>/` path.

### 3 · Repository code is pulled from GitHub at runtime

Cell 1 runs `git clone https://github.com/OttoPrins/thesis-code-final.git` into  
`/kaggle/working/thesis-code/` every time you run it. There is **no `thesis-code-repo`  
Kaggle dataset to attach** — local edits + `git push` are picked up by the next session  
automatically. To pin a specific commit instead of `main`, set `os.environ["THESIS_REF"]`  
to a branch / tag / SHA before running Cell 1.

### 4 · Enable GPU

In the Kaggle notebook editor: **Settings (gear icon) → Accelerator → GPU T4 x2** (or just T4).  
Without this the notebook runs on CPU and will be much slower.

### 5 · Enable Internet access

**Settings → Internet → On** — needed so Cell 1 can run `pip install`.


---
## Cell 1 — Environment Setup

This cell does three things:
1. Installs the small set of packages that Kaggle does **not** pre-install.
2. Sets `KAGGLE_ENV=1` in the process environment so that every subsequent call to  
   `train.py` — including those launched as subprocesses by `run_seeds.py` — automatically  
   redirects paths to `/kaggle/input/` and `/kaggle/working/results/`.
3. Verifies that a GPU is visible to PyTorch.

**Run this cell first, every time you open the notebook.**

In [ ]:
import subprocess, sys, os, shutil
from pathlib import Path

# ── 1. Install missing packages (skipped if already present) ─────────────────
# We do NOT touch numpy here. Kaggle's base image ships NumPy 2.x and ~15
# preinstalled packages (shap, jax, cupy, opencv, pytensor, ...) require >=2.0.
# The codebase was audited 2026-05-13 and is NumPy 2.x compatible.
def _need_install(pkg_name):
    try:
        __import__(pkg_name)
        return False
    except ImportError:
        return True

to_install = []
if _need_install("lifetimes"):  to_install.append("lifetimes>=0.11.3")
if _need_install("openpyxl"):   to_install.append("openpyxl>=3.1.0")

if to_install:
    print(f"Installing: {to_install}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + to_install, check=True)
    print("Done.\n")
else:
    print("All packages already installed.\n")

# ── 2. Set Kaggle environment flag ────────────────────────────────────────────
os.environ["KAGGLE_ENV"] = "1"
print("KAGGLE_ENV=1 set.\n")

# ── 3. Clone (or refresh) the repo from GitHub ────────────────────────────────
# Internet must be ON: Settings → Internet → On. Pin a specific commit by
# setting THESIS_REF before running this cell, e.g. os.environ["THESIS_REF"]="<sha>".
REPO_URL  = "https://github.com/OttoPrins/thesis-code-final.git"
REPO_PATH = Path("/kaggle/working/thesis-code")
REPO_REF  = os.environ.get("THESIS_REF", "main")

if REPO_PATH.exists():
    subprocess.run(["git", "-C", str(REPO_PATH), "fetch", "--all", "--tags", "--quiet"], check=True)
    subprocess.run(["git", "-C", str(REPO_PATH), "checkout", REPO_REF, "--quiet"], check=True)
    subprocess.run(["git", "-C", str(REPO_PATH), "reset", "--hard", f"origin/{REPO_REF}", "--quiet"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_PATH)], check=True)

os.chdir(REPO_PATH)
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))
sha = subprocess.check_output(
    ["git", "-C", str(REPO_PATH), "rev-parse", "--short", "HEAD"]
).decode().strip()
print(f"Repo : {REPO_PATH}  @ {sha}  (ref={REPO_REF})")


# ── 3.5. Ensure raw datasets are accessible ────────────────────────────────────
# CDNOW_sample.txt comes from git; CDNOW_master.txt is optional and fetched from Kaggle when available.
# UCI has a UCI ML Repository fallback URL if the Kaggle dataset upload was skipped.
# TaFeng and Dunnhumby come from Kaggle (pre-mounted symlink or kaggle CLI download).
DATA_ROOT = Path("/kaggle/working/input")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# ── CDNOW: sample from git, optional master from Kaggle ──────────────────
cdnow_dst    = DATA_ROOT / "cdnow-dataset"
cdnow_sample = cdnow_dst / "CDNOW_sample.txt"
cdnow_master = cdnow_dst / "CDNOW_master.txt"
cdnow_mount  = Path("/kaggle/input/cdnow-dataset")
cdnow_dst.mkdir(exist_ok=True)

# CDNOW_sample.txt (2,357-customer 10% sample) is tracked in git — copy from the
# repo. Fall back to the Kaggle dataset mount if the repo copy is missing.
if not cdnow_sample.exists():
    repo_sample = REPO_PATH / "data" / "raw" / "CDNOW_sample.txt"
    if repo_sample.exists():
        shutil.copy(repo_sample, cdnow_sample)
        print("  cdnow-dataset: CDNOW_sample.txt copied from git repo.")
    elif (cdnow_mount / "CDNOW_sample.txt").exists():
        shutil.copy(cdnow_mount / "CDNOW_sample.txt", cdnow_sample)
        print("  cdnow-dataset: CDNOW_sample.txt copied from /kaggle/input/.")
    else:
        print("  cdnow-dataset: WARN — CDNOW_sample.txt not found in repo or /kaggle/input/.")
else:
    print("  cdnow-dataset: CDNOW_sample.txt already present.")

# CDNOW_master.txt (23,570-customer cohort) is gitignored — comes from the Kaggle
# dataset only. Needed for the Valendin et al. (2022) 39x39 replication protocol.
if not cdnow_master.exists():
    if (cdnow_mount / "CDNOW_master.txt").exists():
        shutil.copy(cdnow_mount / "CDNOW_master.txt", cdnow_master)
        print("  cdnow-dataset: CDNOW_master.txt copied from /kaggle/input/.")
    else:
        print("  cdnow-dataset: CDNOW_master.txt not in /kaggle/input/ — trying kaggle download ...")
        try:
            subprocess.run(
                ["kaggle", "datasets", "download", "-d", "ottoprins/cdnow-dataset",
                 "-p", str(cdnow_dst), "--unzip", "--force"],
                check=True, capture_output=True, timeout=180,
            )
        except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as e:
            print(f"  cdnow-dataset: kaggle download fallback failed ({type(e).__name__}).")
        if cdnow_master.exists():
            print("  cdnow-dataset: CDNOW_master.txt downloaded from ottoprins/cdnow-dataset.")
        else:
            print("  cdnow-dataset: WARN — CDNOW_master.txt still missing. "
                  "Cell 2.5 will skip; normal thesis sweeps use CDNOW_sample.txt.")
else:
    print("  cdnow-dataset: CDNOW_master.txt already present.")

# ── UCI: Kaggle download with UCI ML Repo fallback ────────────────────────────
uci_dst = DATA_ROOT / "uci-retail"
if uci_dst.exists():
    print("  uci-retail: already present.")
elif Path("/kaggle/input/uci-retail").exists():
    uci_dst.symlink_to(Path("/kaggle/input/uci-retail"))
    print("  uci-retail: symlinked from /kaggle/input/.")
else:
    print("  uci-retail: not mounted — trying kaggle download ...")
    try:
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", "ottoprins/uci-retail",
             "-p", str(uci_dst), "--unzip"],
            check=True, capture_output=True, timeout=300,
        )
        print("  uci-retail: kaggle download complete.")
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as e:
        print(f"  uci-retail: kaggle download failed ({type(e).__name__}) — fetching from UCI ML Repo ...")
        uci_dst.mkdir(exist_ok=True)
        subprocess.run(["wget", "-q", "-O", "/tmp/uci.zip",
            "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"],
            check=True)
        subprocess.run(["unzip", "-q", "-o", "/tmp/uci.zip", "-d", str(uci_dst)],
            check=True)
        # UCI zip may extract with a slightly different name — normalise it
        for f in sorted(uci_dst.iterdir()):
            if f.suffix in (".xlsx", ".csv") and "retail" in f.name.lower():
                target_name = uci_dst / "online_retail_II.xlsx"
                if not target_name.exists():
                    f.rename(target_name)
                break
        print("  uci-retail: UCI fallback download complete.")

# ── TaFeng and Dunnhumby: standard Kaggle download / symlink ─────────────────
for slug, api_ref in [("tafeng-dataset", "ottoprins/tafeng-dataset"),
                       ("dunnhumby",      "ottoprins/dunnhumby")]:
    target  = DATA_ROOT / slug
    mounted = Path(f"/kaggle/input/{slug}")
    if target.exists():
        print(f"  {slug}: already present.")
    elif mounted.exists():
        target.symlink_to(mounted)
        print(f"  {slug}: symlinked from /kaggle/input/.")
    else:
        print(f"  {slug}: not mounted — downloading ...")
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", api_ref,
             "-p", str(target), "--unzip"],
            check=True,
        )
        print(f"  {slug}: download complete.")

# ── Confirm what's on disk ────────────────────────────────────────────────────
print("\n── Data directory contents ──")
for slug in ["cdnow-dataset", "uci-retail", "tafeng-dataset", "dunnhumby"]:
    d = DATA_ROOT / slug
    if d.exists():
        files = sorted(f.name for f in d.iterdir() if not f.name.startswith("."))[:6]
        print(f"  {slug}: {files}")
    else:
        print(f"  {slug}: [MISSING]")

os.environ["KAGGLE_DATA_ROOT"] = str(DATA_ROOT)
print(f"\nKAGGLE_DATA_ROOT={DATA_ROOT}  — all datasets ready.\n")

# ── 4. Verify GPU (fail fast if incompatible) ─────────────────────────────────
import torch
print(f"\ntorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cap  = torch.cuda.get_device_capability(0)
    print(f"GPU  : {name}  (sm_{cap[0]}{cap[1]})")
    print(f"VRAM : {vram:.1f} GB")
    if cap < (7, 0):
        raise RuntimeError(
            f"\n  {name} has CUDA capability sm_{cap[0]}{cap[1]} "
            f"(PyTorch {torch.__version__} requires sm_70+).\n"
            "  Fix: Settings -> Accelerator -> GPU T4 x1 or T4 x2\n"
            "       then restart the kernel and re-run Cell 1."
        )
    print(f"GPU capability OK (sm_{cap[0]}{cap[1]} >= 7.0).")
else:
    raise RuntimeError(
        "No GPU found. Enable one: Settings -> Accelerator -> GPU T4 x1 (or T4 x2)."
    )

# ── 5. Output directory ───────────────────────────────────────────────────────
Path("/kaggle/working/results").mkdir(parents=True, exist_ok=True)
print("\n/kaggle/working/results/ — ready.")


---
## Cell 2 — Data Path Verification

Before training anything, confirm that Kaggle has mounted your datasets correctly  
and that the data pipeline can actually read the files.

**What to check:** Required datasets should show `✓` below.  
`CDNOW_master.txt` is optional: if it is missing, Cell 2.5 is skipped and the normal thesis sweeps still run with `CDNOW_sample.txt`.

The pipeline validation (`validate_pipelines.py`) does a lightweight end-to-end check:  
it loads the raw file, runs the weekly aggregation, and verifies tensor shapes — without  
starting any training. It takes ~10 seconds per dataset.

In [ ]:
import os
from pathlib import Path

# ── Check dataset mounts ──────────────────────────────────────────────────────
# After Cell 1, KAGGLE_DATA_ROOT=/kaggle/working/input — check there.
# On a fresh interactive session without Cell 1 run, falls back to /kaggle/input
# (where Dunnhumby is still pre-mounted from the UI attachment).
data_root = os.environ.get("KAGGLE_DATA_ROOT", "/kaggle/input")

REQUIRED_FILES = {
    "cdnow-dataset" : ["CDNOW_sample.txt"],
    "uci-retail"    : ["online_retail_II.xlsx"],
    "tafeng-dataset": ["ta_feng_all_months_merged.csv"],
    "dunnhumby"     : ["transaction_data.csv", "hh_demographic.csv"],
}
OPTIONAL_FILES = {
    "cdnow-dataset" : ["CDNOW_master.txt"],  # only needed for Cell 2.5 Valendin master 39x39
}

all_ok = True
optional_ok = True
for slug, expected in REQUIRED_FILES.items():
    p = Path(data_root) / slug
    label = f"{data_root}/{slug}/"
    if not p.exists():
        print(f"✗  {label}  — NOT FOUND  (run Cell 1 first)")
        all_ok = False
        continue
    actual = sorted(f.name for f in p.iterdir())
    missing = [f for f in expected if f not in actual]
    if missing:
        print(f"⚠  {label}  — found but missing: {missing}")
        print(f"   Files present: {actual[:10]}")
        all_ok = False
    else:
        print(f"✓  {label}  — required files present; {len(actual)} file(s): {actual[:5]}")

for slug, expected in OPTIONAL_FILES.items():
    p = Path(data_root) / slug
    label = f"{data_root}/{slug}/"
    if not p.exists():
        optional_ok = False
        print(f"⚠  {label}  — optional files unavailable: {expected}")
        continue
    actual = sorted(f.name for f in p.iterdir())
    missing = [f for f in expected if f not in actual]
    if missing:
        optional_ok = False
        print(f"⚠  {label}  — optional files missing: {missing}")
        print(f"   Cell 2.5 will skip; normal thesis sweeps continue with CDNOW_sample.txt.")
    else:
        print(f"✓  {label}  — optional Valendin master file present.")

print()
if not all_ok:
    raise SystemExit(
        "Missing required Kaggle input files. Run Cell 1 again and make sure the "
        "required Kaggle datasets are mounted or downloadable."
    )
else:
    print("Required datasets ready.")
    if not optional_ok:
        print("Optional Valendin master replication is unavailable; Cell 2.5 will skip.")

# ── Quick pipeline validation (CDNOW only — takes ~10 s) ──────────────────────
# This runs the data pipeline end-to-end and checks tensor shapes.
# It does NOT train a model.
print("\nRunning pipeline validation for CDNOW...")
!python validate_pipelines.py --dataset cdnow

# Uncomment to validate other datasets:
# !python validate_pipelines.py --dataset uci
# !python validate_pipelines.py --dataset tafeng
# !python validate_pipelines.py --dataset dunnhumby


---
## Cell 2.5 — Stage 1 Replication: Valendin et al. (2022) CDNOW master 39×39

Single seed, single config. This is the **literal** paper protocol: 23,570-customer
master cohort, 39-week calibration + 39-week holdout, multinomial autoregressive
sampling, observed frequency classes, ~10 min on T4.

Reference values from the paper (baked into the config under `evaluation:`):
- `freq_rmse ≈ 1.86`
- `bias_pct ≈ −0.7%`
- `freq_valendin_mape ≈ 13.8%`

**Run this BEFORE the multi-seed sweeps when `CDNOW_master.txt` is available.**
If the master file is missing, this cell prints a skip message and the normal
sample-based thesis sweeps continue. If the assertion cell that follows reports
a mismatch after an actual master run, halt and diagnose before launching the full sweep.


In [ ]:
# ── Stage 1 — Valendin et al. (2022) CDNOW master 39×39 replication ──────────
import os, subprocess, sys
from pathlib import Path

data_root = Path(os.environ.get("KAGGLE_DATA_ROOT", "/kaggle/input"))
cdnow_master = data_root / "cdnow-dataset" / "CDNOW_master.txt"
if not cdnow_master.exists():
    visible = []
    cdnow_dir = cdnow_master.parent
    if cdnow_dir.exists():
        visible = sorted(p.name for p in cdnow_dir.iterdir() if not p.name.startswith("."))
    VALENDIN_MASTER_AVAILABLE = False
    print(
        f"SKIP Cell 2.5: Valendin master 39x39 requires {cdnow_master}.\n"
        f"Visible files: {visible or '<none>'}.\n"
        "Normal thesis sweeps continue with CDNOW_sample.txt. To enable this optional "
        "master sanity check, re-run upload_data_to_kaggle.sh locally, then "
        "bash push_to_kaggle.sh and hard-refresh this Kaggle notebook."
    )
else:
    VALENDIN_MASTER_AVAILABLE = True
    subprocess.run([
        sys.executable, "train.py",
        "--config", "experiments/configs/lstm_base_cdnow_valendin_master_39x39.yaml",
        "--kaggle",
        "--seed_override", "42",
    ], check=True)




---
## Cell 2.6 — Stage 1 Assertion: did we replicate Valendin's numbers?

Loads the metrics JSON produced by the previous cell and compares each of the
three primary frequency metrics against the reference values stored under
`evaluation:` in the config. If the optional master run was skipped or no metrics
were produced, this cell skips without blocking the normal thesis sweeps.


In [ ]:
# ── Stage 1 assertion — compare run metrics against config-stored reference values ─
import json, yaml
from pathlib import Path

cfg_path = Path("experiments/configs/lstm_base_cdnow_valendin_master_39x39.yaml")
results_tables = Path("/kaggle/working/results/tables")
metrics_candidates = [
    results_tables / "lstm_base_cdnow_valendin_master_39x39_seed42_metrics.json",
    results_tables / "lstm_base_cdnow_valendin_master_39x39_seed42_sample_metrics.json",
]
globbed_candidates = sorted(
    results_tables.glob("lstm_base_cdnow_valendin_master_39x39_seed42*_metrics.json"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
) if results_tables.exists() else []
metrics_path = next((p for p in metrics_candidates + globbed_candidates if p.exists()), None)

if metrics_path is None and globals().get("VALENDIN_MASTER_AVAILABLE") is False:
    print("SKIP Cell 2.6: Cell 2.5 was skipped because CDNOW_master.txt is unavailable.")
elif metrics_path is None:
    print(
        "SKIP Cell 2.6: no Valendin master metrics file was found. "
        "This optional sanity check will not block the normal thesis sweeps."
    )
else:
    print(f"Using metrics file: {metrics_path}")
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    ref = cfg.get("evaluation", {})
    ref_rmse = float(ref["valendin_reference_rmse"])
    ref_bias = float(ref["valendin_reference_bias_pct"])
    ref_mape = float(ref["valendin_reference_mape"])

    with open(metrics_path) as f:
        m = json.load(f)
    got_rmse = float(m["freq_rmse"])
    got_bias = float(m["bias_pct"])
    got_mape = float(m["freq_valendin_mape"])

    # tolerances: ±20% on RMSE, ±5 percentage points on bias and MAPE
    rmse_lo, rmse_hi = 0.8 * ref_rmse, 1.2 * ref_rmse
    bias_lo, bias_hi = ref_bias - 5.0, ref_bias + 5.0
    mape_lo, mape_hi = ref_mape - 5.0, ref_mape + 5.0

    def status(value, lo, hi):
        return "PASS" if lo <= value <= hi else "FAIL"

    print(f"Stage-1 replication check vs Valendin et al. (2022) reference values")
    print(f"  freq_rmse           : {got_rmse:7.3f}  ref {ref_rmse:5.2f}  range [{rmse_lo:5.2f}, {rmse_hi:5.2f}]  → {status(got_rmse, rmse_lo, rmse_hi)}")
    print(f"  bias_pct            : {got_bias:+7.2f}%  ref {ref_bias:+5.2f}%  range [{bias_lo:+5.2f}, {bias_hi:+5.2f}]  → {status(got_bias, bias_lo, bias_hi)}")
    print(f"  freq_valendin_mape  : {got_mape:7.2f}%  ref {ref_mape:5.2f}%  range [{mape_lo:5.2f}, {mape_hi:5.2f}]  → {status(got_mape, mape_lo, mape_hi)}")
    print()
    if all([
        rmse_lo <= got_rmse <= rmse_hi,
        bias_lo <= got_bias <= bias_hi,
        mape_lo <= got_mape <= mape_hi,
    ]):
        print("✓ Stage-1 replication PASSED — safe to launch the multi-seed sweeps.")
    else:
        print("✗ Stage-1 replication FAILED — halt and diagnose BEFORE running Cell 3 and onward.")


---
## Cell 3 — Training: CDNOW + UCI  (~3–5 h on T4 × 3 seeds)

This cell runs the full 3-seed sweep for CDNOW and UCI using `run_seeds.py`.

**Config track:** v6 (base models) / v6b (joint + transformer on sparse datasets).
- Base LSTM: v6 — noshrink aggregate calibration (`aggregate_shrinkage: 1.0`)
- Joint LSTM + Transformer: v6b — additionally pins `freq_logvar_max: 0.0` to prevent
  Kendall uncertainty weighting from collapsing the frequency task on sparse data
  (root cause: on CDNOW/UCI, predicting zero is cheap; the Kendall optimizer drifted
  freq log_var → 2.0, reducing freq weight to exp(−2)/2 ≈ 0.07)

**Inference:** `sample` mode, 30 scenarios (~10 min per run on T4)  
**Seeds:** 42, 7, 2024 (3 × 6 configs = 18 runs total)  
**`--skip_existing`** resumes from where you left off if the session is interrupted.

**Expected output for each run:**
```
Metrics saved: /kaggle/working/results/tables/lstm_base_cdnow_v6_seed42_sample_metrics.json
```

In [ ]:
# ── CDNOW + UCI v6/v6b: 6 configs × 3 seeds = 18 runs ──────────────────────
# v6:  base models — noshrink calibration, sample inference
# v6b: joint/transformer — v6 + freq_logvar_max=0.0 (Kendall collapse fix)
# --skip_existing resumes from where the session left off.
!python run_seeds.py \
    --configs lstm_base_cdnow_v6     lstm_joint_cdnow_v6b     transformer_joint_cdnow_v6b \
              lstm_base_uci_v6       lstm_joint_uci_v6b       transformer_joint_uci_v6b \
    --seeds 42 7 2024 \
    --modes sample \
    --skip_existing

---
## Cell 4 — Training: Dunnhumby  (~2–4 h on T4 × 3 seeds)

Run this after Cell 3 completes (or in a second session using `--skip_existing`).  
Check the session timer in the Kaggle editor — you need at least ~2 h free.

**Config track:** v6 for all Dunnhumby models (Dunnhumby is a dense dataset;
the Kendall collapse pathology does not occur — no v6b fix needed).  
**Seeds:** 42, 7, 2024 (3 × 3 configs = 9 runs total)

In [ ]:
# ── Dunnhumby v6: 3 configs × 3 seeds = 9 runs ──────────────────────────────
# Dense dataset — v6 (no Kendall fix needed; collapse doesn't occur on dense data)
!python run_seeds.py \
    --configs lstm_base_dunnhumby_v6 lstm_joint_dunnhumby_v6 transformer_joint_dunnhumby_v6 \
    --seeds 42 7 2024 \
    --modes sample \
    --skip_existing

---
## Cell 5 — Training: TaFeng  (~5–8 h on T4 × 3 seeds, run LAST)

TaFeng has ~32,000 customers — the largest dataset. Run this last.  
If you run out of session time, re-open the notebook, run Cell 1, then re-run this
cell — `--skip_existing` skips already-completed runs automatically.

**Config track:** v6 (base) / v6b (joint + transformer) — same fix as CDNOW/UCI.  
**Seeds:** 42, 7, 2024 (3 × 3 configs = 9 runs total)

In [ ]:
# ── TaFeng v6/v6b: 3 configs × 3 seeds = 9 runs ─────────────────────────────
# v6b joint/transformer: freq_logvar_max=0.0 fix (same sparse-data pathology as CDNOW/UCI)
!python run_seeds.py \
    --configs lstm_base_tafeng_v6 lstm_joint_tafeng_v6b transformer_joint_tafeng_v6b \
    --seeds 42 7 2024 \
    --modes sample \
    --skip_existing

# ── OOM fallback — retry with smaller batch if CUDA out of memory ─────────────
# !python run_seeds.py \
#     --configs lstm_base_tafeng_v6 lstm_joint_tafeng_v6b transformer_joint_tafeng_v6b \
#     --seeds 42 7 2024 --modes sample --skip_existing \
#     --batch_size_override 64

## CDNOW 39×39 sensitivity check

Sensitivity test using the original Valendin et al. (2022) calibration window (39 calibration + 39 holdout weeks) on the same 2,357-customer Wharton CDNOW sample. Reported in the thesis alongside the 52+26 headline.

In [ ]:
# ── CDNOW 39+39 sensitivity check — single config × 1 seed = 1 run ─────────
# Reported as robustness check vs the 52+26 headline (per methodology .tex).
!python run_seeds.py \
    --configs lstm_base_cdnow_39x39_v4_soft \
    --seeds 42 \
    --modes expected \
    --skip_existing

---
## Cell 6 — Single targeted run  (debug / one-off)

Use this cell to run a single config directly — useful for debugging, checking that  
paths are correct, or running a quick smoke test before committing to the full sweep.

The `--kaggle` flag here is redundant (we already set `KAGGLE_ENV=1` in Cell 1), but  
it is included as an explicit reminder that the override is active.

**Smoke test** (3 epochs, 2 scenarios — completes in ~1 minute):

In [ ]:
# ── Smoke test — v6 base CDNOW, 3 epochs, 1 scenario (~1 min) ────────────────
!python train.py \
    --config experiments/configs/lstm_base_cdnow_v6.yaml \
    --kaggle \
    --seed_override 42 \
    --max_epochs 3 \
    --n_scenarios 1 \
    --diagnostic_only

# ── Full single run (one config, one seed) ────────────────────────────────────
# !python train.py \
#     --config experiments/configs/transformer_joint_cdnow_v6b.yaml \
#     --kaggle \
#     --seed_override 42

---
## Cells ext3.1–ext3.3 — Extension 3: Covariate Ablation (Stage 4)

These cells complete Stage 4 of the thesis (Dunnhumby covariate attribution).
Extension 3 is split into smaller resumable groups so a Kaggle session can archive
partial outputs after each group instead of losing a long-running cell.

| Variant | Static covariates | Dynamic covariates |
|---------|------------------|--------------------|
| `none`  | — | — |
| `static` | income, household size | — |
| `dynamic` | — | coupon redemptions, campaign flag |
| `full`  | income, household size | coupon redemptions, campaign flag |

`--skip_existing` makes each group resumable. Completed strict or repaired runs are left alone.
Run ext3.1 first, then ext3.2, then SHAP. Each training group refreshes `results_archive.zip`.


In [ ]:
# ── ext3.1 — Train Extension 3 LSTM v4_soft covariate-ablation variants ─────
# 4 configs × 1 seed = 4 runs. --skip_existing makes this resumable.
import subprocess, sys

def _archive_results(label):
    import shutil
    from pathlib import Path
    results_dir = Path("/kaggle/working/results")
    archive_stem = f"/kaggle/working/results_archive_{label}"
    if results_dir.exists() and any(results_dir.rglob("*")):
        shutil.make_archive(archive_stem, "zip", results_dir)
        archive_path = Path(archive_stem + ".zip")
        print(f"Archive refreshed: {archive_path} ({archive_path.stat().st_size / (1024 ** 2):.1f} MB)")

EXT3_LSTM_CONFIGS = [
    "extension3_lstm_none_dunnhumby_v4_soft",
    "extension3_lstm_static_dunnhumby_v4_soft",
    "extension3_lstm_dynamic_dunnhumby_v4_soft",
    "extension3_lstm_full_dunnhumby_v4_soft",
]
subprocess.run([
    sys.executable, "run_seeds.py",
    "--configs", *EXT3_LSTM_CONFIGS,
    "--seeds", "42",
    "--modes", "expected",
    "--skip_existing",
    "--heartbeat_interval", "60",
], check=False)
_archive_results("ext3_lstm_v4_soft")
print("\nExtension 3 LSTM v4_soft covariate ablation group complete.")

---
## Cell ext3.2 — Extension 3 Transformer Variants

Run this after ext3.1. Transformer Extension 3 runs are separated from LSTM runs so
any long-running instability is easier to resume and diagnose.


In [ ]:
# ── ext3.2 — Train Extension 3 Transformer v4_soft covariate-ablation variants ─
# 4 configs × 1 seed = 4 runs. --skip_existing makes this resumable.
import subprocess, sys

def _archive_results(label):
    import shutil
    from pathlib import Path
    results_dir = Path("/kaggle/working/results")
    archive_stem = f"/kaggle/working/results_archive_{label}"
    if results_dir.exists() and any(results_dir.rglob("*")):
        shutil.make_archive(archive_stem, "zip", results_dir)
        archive_path = Path(archive_stem + ".zip")
        print(f"Archive refreshed: {archive_path} ({archive_path.stat().st_size / (1024 ** 2):.1f} MB)")

EXT3_TRANSFORMER_CONFIGS = [
    "extension3_transformer_none_dunnhumby_v4_soft",
    "extension3_transformer_static_dunnhumby_v4_soft",
    "extension3_transformer_dynamic_dunnhumby_v4_soft",
    "extension3_transformer_full_dunnhumby_v4_soft",
]
subprocess.run([
    sys.executable, "run_seeds.py",
    "--configs", *EXT3_TRANSFORMER_CONFIGS,
    "--seeds", "42",
    "--modes", "expected",
    "--skip_existing",
    "--heartbeat_interval", "60",
], check=False)
_archive_results("ext3_transformer_v4_soft")
print("\nExtension 3 Transformer v4_soft covariate ablation group complete.")

---
## Cell ext3.3 — SHAP Covariate Attribution

Runs `src/evaluation/shap_analysis.py` on the best `extension3_lstm_full` checkpoint
(seed 42). Outputs covariate importance bar charts to `results/plots/`.

**Requires:** ext3.1 complete and `extension3_lstm_full_dunnhumby_final_seed42_sample*.pt` present.


In [ ]:
# ── ext3.3 — SHAP covariate attribution (v4_soft full Joint LSTM) ────────────
import subprocess, sys, glob
from pathlib import Path

# Locate the v4_soft full-covariate checkpoint (seed 42)
ckpt_pattern = "/kaggle/working/results/checkpoints/extension3_lstm_full_dunnhumby_v4_soft*seed42*.pt"
ckpts = sorted(glob.glob(ckpt_pattern))
if not ckpts:
    print(f"No checkpoint found at {ckpt_pattern} — run ext3.1 first.")
else:
    Path("/kaggle/working/results/plots").mkdir(parents=True, exist_ok=True)
    subprocess.run([
        sys.executable, "-m", "src.evaluation.shap_analysis",
        "--config", "experiments/configs/extension3_lstm_full_dunnhumby_v4_soft.yaml",
        "--checkpoint", ckpts[-1],
        "--n_background", "100",
        "--n_explain", "200",
        "--out_dir", "/kaggle/working/results/plots",
    ], check=False)
    plots = sorted(Path("/kaggle/working/results/plots").glob("*.png"))
    print(f"\nSHAP plots saved: {len(plots)}")
    for p in plots:
        print(f"  {p.name}")

---
## Cell 8 — Generate Thesis Figures And Result Tables

Builds strict comparison tables, repaired-variant robustness tables, failure appendix,
comparison bar charts, seed aggregates, and the LaTeX table from final-manifest metrics only.
Run this after all training cells have completed (or after downloading partial results).

Outputs land in `/kaggle/working/results/tables/` and `/kaggle/working/results/plots/`
and are included in the archive automatically (re-run Cell 9 afterwards to refresh the zip).


In [ ]:
# ── Cell 8 — Thesis figures + LaTeX table from final-manifest runs ───────────
# v4_soft uses expected-value inference, so we MUST pass --include_expected.
# --protocol_variant all includes both strict and repaired runs.
import subprocess
from pathlib import Path

Path("/kaggle/working/results/plots").mkdir(parents=True, exist_ok=True)
subprocess.run([
    "python", "-m", "src.evaluation.compare",
    "--results_dir", "/kaggle/working/results",
    "--latex", "--plots", "--seeds",
    "--include_expected",
    "--protocol_variant", "all",
], check=False)

plots = (
    sorted(Path("/kaggle/working/results/plots").glob("*.png")) +
    sorted(Path("/kaggle/working/results/plots").glob("*.pdf"))
)
print(f"Plots generated: {len(plots)}")
for p in plots:
    print(f"  {p.name}")
print("\nNow run Cell 9 to refresh results_archive.zip.")

---
## Cell 9 — Archive results for download

**Always run this cell before the session ends**, even if training is still running  
in other cells — it archives whatever has been saved so far.

After running:
1. In the Kaggle notebook editor, click **"Save & Run All"** (or just save).
2. Go to the **Output** tab (right panel or bottom of the page).
3. Find `results_archive.zip` and click **Download**.

The archive contains:
- `tables/` — `*_metrics.json` and `*_history.json` for every completed run
- `checkpoints/` — `.pt` model weights
- `plots/` — any figures generated by the evaluation scripts

> **Note:** Kaggle also shows individual output files under the Output tab.  
> The zip just makes bulk downloading easier.

In [ ]:
import shutil, os
from pathlib import Path

results_dir  = Path("/kaggle/working/results")
archive_stem = "/kaggle/working/results_archive"   # .zip will be appended automatically

if not results_dir.exists() or not any(results_dir.rglob("*")):
    print("No results found yet — run at least one training cell first.")
else:
    # Report what we have before archiving
    metrics_files = sorted(results_dir.rglob("*_metrics.json"))
    ckpt_files    = sorted(results_dir.rglob("*.pt"))
    print(f"Metrics files  : {len(metrics_files)}")
    print(f"Checkpoints    : {len(ckpt_files)}")
    for f in metrics_files:
        print(f"  {f.name}")

    # Build zip
    shutil.make_archive(archive_stem, "zip", results_dir)
    archive_path = Path(archive_stem + ".zip")
    size_mb = archive_path.stat().st_size / (1024 ** 2)
    print(f"\nArchive created : {archive_path}  ({size_mb:.1f} MB)")
    print("Download via    : Kaggle notebook → Output tab → results_archive.zip")

---
## Troubleshooting

### Notebook on Kaggle still shows old cells after `bash push_to_kaggle.sh`
The Kaggle web editor caches the notebook revision in your browser tab. After  
pushing, **hard-refresh the tab** (⌘⇧R on macOS, Ctrl-Shift-R on Linux/Windows)  
before clicking Run All. Otherwise you re-run the previously cached version.

### `git clone` fails in Cell 1
Internet is off. Open **Settings → Internet → On**, then re-run Cell 1.


### `RuntimeError: … sm_60 … requires sm_70+`
Kaggle allocated a P100 (Pascal) GPU, which PyTorch 2.x no longer supports.
Fix: **Settings → Accelerator → GPU T4 x1** (or GPU T4 x2), then restart the
kernel and re-run Cell 1. T4 (sm_75) is well-supported.

### Cell 2.5 says `SKIP` because `CDNOW_master.txt` is missing
That is safe for the normal thesis sweeps: they use `CDNOW_sample.txt`. Cell 2.5 is only the
optional Valendin master 39×39 sanity check. To enable it, make sure `cdnow-dataset` contains
both `CDNOW_sample.txt` and `CDNOW_master.txt`, then re-run `bash upload_data_to_kaggle.sh`,
`bash push_to_kaggle.sh`, hard-refresh the Kaggle tab, and re-run Cells 1 and 2.

### `FileNotFoundError: CDNOW raw file not found in /kaggle/input/cdnow-dataset`
If the whole CDNOW folder is missing, the Kaggle dataset slug doesn't match what the code expects.
The slug mapping is defined in `src/utils/config.py` → `_KAGGLE_SLUG_MAP`. Expected slugs:
`cdnow-dataset`, `uci-retail`, `tafeng-dataset`, `dunnhumby`. If you used a different title
when uploading, either re-create the dataset with the correct title, or override the path in Cell 1:
```python
import os
os.environ["KAGGLE_DATA_ROOT"] = "/kaggle/input/my-actual-slug"
os.environ["KAGGLE_ENV"] = "1"
```

### `CUDA out of memory` on TaFeng
TaFeng has ~32 k customers. Re-run the TaFeng cell with the built-in batch-size override:
```bash
!python run_seeds.py \
    --configs lstm_base_tafeng_v2 lstm_joint_tafeng_v2 transformer_joint_tafeng_v2 \
    --seeds 42 --modes sample --skip_existing \
    --batch_size_override 64
```

### `ModuleNotFoundError: No module named 'src'`
`os.chdir(REPO_PATH)` in Cell 1 must point at the directory that contains `src/` and `train.py`.  
Double-check `REPO_PATH` and that the repository files were uploaded correctly.

### Session times out mid-sweep
`run_seeds.py --skip_existing` resumes from where it left off — just re-open the notebook,  
run Cell 1 (to set `KAGGLE_ENV=1` again), then re-run the relevant training cell.  
Completed runs are detected by their existing `*_metrics.json` files and skipped automatically.

### `rpy2` / `cmdstanpy` import errors
These are **not needed** for DL training. If you accidentally call `run_benchmarks.py`  
on Kaggle it will fail — that script requires R and Stan. Only run it locally.  
The cells in this notebook never call `run_benchmarks.py`.

### Probabilistic benchmarks (Pareto/NBD, Pareto/GGG, GPPM)
These cannot run on Kaggle (missing R / Stan). Run them locally with your full environment  
and merge the resulting `*_metrics.json` files into the same `results/tables/` directory  
before generating comparison tables.
